In [ ]:
import sys
import re
import numpy as np
import pandas as pd
import openpyxl
from datetime import date

print('=' * 60, file=sys.stderr)
print('Waste Processing Plant Financial Model', file=sys.stderr)
print('=' * 60, file=sys.stderr)

# ======================================================
# 1. LOAD DATA FROM EXCEL
# ======================================================
DATA_DIR = '/workspace/data'
xlsx_path = f'{DATA_DIR}/MO17-Round-1-Sec-4-Go-With-The-Flow-data.xlsx'

wb = openpyxl.load_workbook(xlsx_path, data_only=True)
print(f'Sheet names: {wb.sheetnames}', file=sys.stderr)

# Find the inputs sheet
inputs_ws = None
for name in wb.sheetnames:
    if 'input' in name.lower() or 'assumption' in name.lower():
        inputs_ws = wb[name]
        break
if inputs_ws is None:
    inputs_ws = wb[wb.sheetnames[0]]

print(f'Using sheet: {inputs_ws.title}', file=sys.stderr)
print(f'Max rows: {inputs_ws.max_row}, Max cols: {inputs_ws.max_column}', file=sys.stderr)

# Print all non-empty cells to understand the layout
all_data = {}
for row in inputs_ws.iter_rows(min_row=1, max_row=inputs_ws.max_row, values_only=False):
    for cell in row:
        if cell.value is not None:
            all_data[cell.coordinate] = cell.value

# Find rows containing 'waste' or 'tonnage' to locate the waste received data
waste_row = None
for row_idx in range(1, inputs_ws.max_row + 1):
    for col_idx in range(1, min(inputs_ws.max_column + 1, 12)):
        cell = inputs_ws.cell(row=row_idx, column=col_idx)
        if cell.value and isinstance(cell.value, str) and 'total waste' in cell.value.lower():
            waste_row = row_idx
            print(f'Found "total waste" at row {row_idx}, col {col_idx}: {cell.value}', file=sys.stderr)
            break
    if waste_row:
        break

# Also search for specific patterns
for row_idx in range(1, inputs_ws.max_row + 1):
    for col_idx in range(1, min(inputs_ws.max_column + 1, 12)):
        cell = inputs_ws.cell(row=row_idx, column=col_idx)
        if cell.value and isinstance(cell.value, str):
            val_lower = cell.value.lower().strip()
            if any(k in val_lower for k in ['waste', 'tonnage', 'received', 'period', 'quarter']):
                print(f'  Row {row_idx}, Col {col_idx}: {cell.value}', file=sys.stderr)

# Print row-by-row for full inspection
print('\n--- Full sheet contents ---', file=sys.stderr)
for row_idx in range(1, inputs_ws.max_row + 1):
    row_vals = []
    for col_idx in range(1, inputs_ws.max_column + 1):
        cell = inputs_ws.cell(row=row_idx, column=col_idx)
        if cell.value is not None:
            row_vals.append((col_idx, cell.value))
    if row_vals:
        print(f'Row {row_idx}: {row_vals}', file=sys.stderr)

# Also check other sheets
for sn in wb.sheetnames:
    if sn != inputs_ws.title:
        ws2 = wb[sn]
        print(f'\n--- Sheet: {sn} ---', file=sys.stderr)
        for row_idx in range(1, min(ws2.max_row + 1, 50)):
            row_vals = []
            for col_idx in range(1, min(ws2.max_column + 1, 70)):
                cell = ws2.cell(row=row_idx, column=col_idx)
                if cell.value is not None:
                    row_vals.append((col_idx, cell.value))
            if row_vals:
                print(f'Row {row_idx}: {row_vals}', file=sys.stderr)

In [ ]:
# ======================================================
# 2. EXTRACT QUARTERLY WASTE TONNAGE DATA
# ======================================================
# The Excel file has a layout with period dates in header rows.
# From the calculations_sheet.csv, we know:
#   Row 6 has Period Start dates starting from a specific column
#   Row 7 has Period End dates, with a "Sum" column before the data
# The data columns correspond to quarters.
# We need to find the "Total Waste Received" row and extract quarterly values.

import datetime as dt

waste_data_quarterly = None
data_columns = None

# Step 1: Find the period header row and identify data columns
for sn in wb.sheetnames:
    ws = wb[sn]
    for row_idx in range(1, min(ws.max_row + 1, 20)):
        for col_idx in range(1, min(ws.max_column + 1, 15)):
            cell = ws.cell(row=row_idx, column=col_idx)
            if cell.value and isinstance(cell.value, str) and 'period' in cell.value.lower():
                print(f'Found period header in "{sn}" row {row_idx}, col {col_idx}: {cell.value}', file=sys.stderr)
                # Find all date-valued columns in this row or the next
                for check_row in [row_idx, row_idx + 1]:
                    date_cols = []
                    for c in range(col_idx, ws.max_column + 1):
                        v = ws.cell(row=check_row, column=c).value
                        if isinstance(v, (dt.datetime, dt.date)):
                            date_cols.append(c)
                    if len(date_cols) >= 50:
                        data_columns = date_cols
                        print(f'  Found {len(date_cols)} date columns in row {check_row}, cols {date_cols[0]}-{date_cols[-1]}', file=sys.stderr)
                        # Print first few dates
                        for dc in date_cols[:5]:
                            print(f'    Col {dc}: {ws.cell(row=check_row, column=dc).value}', file=sys.stderr)
                        break
                if data_columns:
                    break
        if data_columns:
            break
    if data_columns:
        break

# Step 2: Search for waste data row
for sn in wb.sheetnames:
    ws = wb[sn]
    for row_idx in range(1, ws.max_row + 1):
        for col_idx in range(1, min(ws.max_column + 1, 15)):
            cell = ws.cell(row=row_idx, column=col_idx)
            if cell.value and isinstance(cell.value, str):
                val_lower = cell.value.lower().strip()
                if 'total waste' in val_lower or ('waste' in val_lower and 'received' in val_lower):
                    print(f'Found waste label in "{sn}" row {row_idx}, col {col_idx}: {cell.value}', file=sys.stderr)
                    
                    if data_columns:
                        # Extract values from known data columns
                        vals = []
                        for c in data_columns:
                            v = ws.cell(row=row_idx, column=c).value
                            if isinstance(v, (int, float)):
                                vals.append(v)
                            else:
                                vals.append(0)  # Handle None as 0
                        print(f'  Extracted {len(vals)} values using data columns', file=sys.stderr)
                        print(f'  First 5: {vals[:5]}', file=sys.stderr)
                        if any(v > 10000 for v in vals[:5]):
                            waste_data_quarterly = vals
                            break
                    else:
                        # Fallback: extract all large numeric values from the row
                        vals = []
                        for c in range(1, ws.max_column + 1):
                            v = ws.cell(row=row_idx, column=c).value
                            if isinstance(v, (int, float)) and v > 1000:  # filter small numbers
                                vals.append((c, v))
                        if len(vals) >= 50:
                            waste_data_quarterly = [v for _, v in vals]
                            print(f'  Extracted {len(vals)} values (fallback)', file=sys.stderr)
                            print(f'  First 5: {waste_data_quarterly[:5]}', file=sys.stderr)
                            break
        if waste_data_quarterly:
            break
    if waste_data_quarterly:
        break

# Strategy 3: Look for rows with many large numbers if label search failed
if waste_data_quarterly is None:
    print('Label search failed, trying large-number strategy...', file=sys.stderr)
    best_row_info = None
    best_count = 0
    for sn in wb.sheetnames:
        ws = wb[sn]
        for row_idx in range(1, ws.max_row + 1):
            if data_columns:
                vals = []
                for c in data_columns:
                    v = ws.cell(row=row_idx, column=c).value
                    if isinstance(v, (int, float)) and v > 50000:
                        vals.append(v)
                if len(vals) > best_count:
                    best_count = len(vals)
                    best_row_info = (sn, row_idx, [ws.cell(row=row_idx, column=c).value for c in data_columns])
            else:
                vals = []
                for col_idx in range(1, ws.max_column + 1):
                    v = ws.cell(row=row_idx, column=col_idx).value
                    if isinstance(v, (int, float)) and v > 50000:
                        vals.append(v)
                if len(vals) > best_count:
                    best_count = len(vals)
                    best_row_info = (sn, row_idx, vals)
    
    if best_row_info and best_count >= 20:
        sn, row_idx, vals = best_row_info
        print(f'Found potential waste row in "{sn}" row {row_idx}: {best_count} large values', file=sys.stderr)
        ws = wb[sn]
        for c in range(1, 12):
            label = ws.cell(row=row_idx, column=c).value
            if label:
                print(f'  Label at col {c}: {label}', file=sys.stderr)
        # If using data_columns, vals might contain None values
        waste_data_quarterly = [v if isinstance(v, (int, float)) else 0 for v in vals]
        print(f'  First 5 values: {waste_data_quarterly[:5]}', file=sys.stderr)

if waste_data_quarterly is None:
    raise ValueError('Could not find waste data in the Excel workbook')

print(f'\nRaw waste data: {len(waste_data_quarterly)} values', file=sys.stderr)
print(f'  First 8: {waste_data_quarterly[:8]}', file=sys.stderr)
print(f'  Last 4: {waste_data_quarterly[-4:]}', file=sys.stderr)

In [ ]:
# ======================================================
# 3. BUILD THE QUARTERLY MODEL
# ======================================================
# Quarterly periods: Q1 2018 through Q4 2030 = 52 quarters

import calendar

quarters = []
for year in range(2018, 2031):
    for q in range(1, 5):
        m_start = (q - 1) * 3 + 1
        m_end = q * 3
        q_start = date(year, m_start, 1)
        q_end = date(year, m_end, calendar.monthrange(year, m_end)[1])
        quarters.append({
            'year': year,
            'q': q,
            'q_start': q_start,
            'q_end': q_end,
            'label': f'Q{q} {year}',
        })

n_quarters = len(quarters)  # 52
print(f'Model has {n_quarters} quarters: {quarters[0]["label"]} to {quarters[-1]["label"]}', file=sys.stderr)

# Map the extracted waste data to our 52 quarters.
# The Excel may have 52 values (Q1 2018 - Q4 2030) or 53 (Dec 2017 + 52 quarters).
# It might also have a "Sum" column.

n_raw = len(waste_data_quarterly)
print(f'Raw waste data count: {n_raw}', file=sys.stderr)

if n_raw == 52:
    total_waste_received = list(waste_data_quarterly)
elif n_raw == 53:
    # First value might be Dec 2017 (period 0) or a Sum column
    # Check if first value looks like a sum (much larger than individual quarters)
    if waste_data_quarterly[0] > waste_data_quarterly[1] * 10:
        # First value is likely a sum, skip it and take next 52
        total_waste_received = list(waste_data_quarterly[1:53])
        print(f'  Skipping first value (likely sum): {waste_data_quarterly[0]:,.0f}', file=sys.stderr)
    else:
        # First value is Dec 2017 (period 0), skip it
        total_waste_received = list(waste_data_quarterly[1:53])
        print(f'  Skipping first value (Dec 2017): {waste_data_quarterly[0]:,.0f}', file=sys.stderr)
elif n_raw == 54:
    # Might have sum + Dec 2017 + 52 quarters, or Dec 2017 + 52 + extra
    # Try skipping first two
    total_waste_received = list(waste_data_quarterly[2:54])
    print(f'  Skipping first two values: {waste_data_quarterly[0]:,.0f}, {waste_data_quarterly[1]:,.0f}', file=sys.stderr)
elif n_raw > 52:
    # Try to find the 52 quarterly values
    # The quarterly data should start from a specific column
    # Take the last 52 values as a fallback
    total_waste_received = list(waste_data_quarterly[-52:])
    print(f'  Using last 52 of {n_raw} values', file=sys.stderr)
else:
    raise ValueError(f'Expected at least 52 quarterly values, got {n_raw}')

assert len(total_waste_received) == 52, f'Expected 52 quarterly values, got {len(total_waste_received)}'

# Sanity check: waste values should be > 0 and reasonable (thousands to hundreds of thousands)
for i, w in enumerate(total_waste_received):
    if w <= 0:
        print(f'WARNING: Quarter {i} ({quarters[i]["label"]}) has waste = {w}', file=sys.stderr)

print(f'\nTotal waste received per quarter loaded: {len(total_waste_received)} quarters', file=sys.stderr)
print(f'  Q1 2018: {total_waste_received[0]:,.0f} tonnes', file=sys.stderr)
print(f'  Q4 2030: {total_waste_received[-1]:,.0f} tonnes', file=sys.stderr)
print(f'  Min: {min(total_waste_received):,.0f}, Max: {max(total_waste_received):,.0f}', file=sys.stderr)
print(f'  Total sum: {sum(total_waste_received):,.0f} tonnes', file=sys.stderr)

In [ ]:
# ======================================================
# 4. WASTE FLOW ALLOCATION
# ======================================================
# From the flowchart:
#   Total Waste Received -> 5% Third Party, 95% Authority
#   Authority waste: up to 75,000 -> Guaranteed, remainder -> Additional
#
# Disposal splits:
#   Third party:  5% recycled, 70% incinerated, 25% landfill
#   Guaranteed:   5% recycled, 75% incinerated, 20% landfill
#   Additional:   5% recycled, 75% incinerated, 20% landfill
#
# Recycling split: 0.01% Silver, 0.09% Copper, 99.9% Iron

# Waste categorization
third_party = [t * 0.05 for t in total_waste_received]
authority = [t * 0.95 for t in total_waste_received]
guaranteed = [min(a, 75000) for a in authority]
additional_actual = [a - g for a, g in zip(authority, guaranteed)]

# Q1: Total additional waste received
total_additional = sum(additional_actual)
print(f'Total additional waste received: {total_additional:,.0f} tonnes', file=sys.stderr)

# Disposal allocation per waste stream
# Third party: 5% recycling, 70% incineration, 25% landfill
tp_recycled = [t * 0.05 for t in third_party]
tp_incinerated = [t * 0.70 for t in third_party]
tp_landfill = [t * 0.25 for t in third_party]

# Guaranteed: 5% recycling, 75% incineration, 20% landfill
gw_recycled = [g * 0.05 for g in guaranteed]
gw_incinerated = [g * 0.75 for g in guaranteed]
gw_landfill = [g * 0.20 for g in guaranteed]

# Additional: 5% recycling, 75% incineration, 20% landfill
aw_recycled = [a * 0.05 for a in additional_actual]
aw_incinerated = [a * 0.75 for a in additional_actual]
aw_landfill = [a * 0.20 for a in additional_actual]

# Totals per disposal type
total_recycled = [tp + gw + aw for tp, gw, aw in zip(tp_recycled, gw_recycled, aw_recycled)]
total_incinerated = [tp + gw + aw for tp, gw, aw in zip(tp_incinerated, gw_incinerated, aw_incinerated)]
total_landfill = [tp + gw + aw for tp, gw, aw in zip(tp_landfill, gw_landfill, aw_landfill)]

# Recycling sub-split: 0.01% Silver, 0.09% Copper, 99.9% Iron
silver_tonnes = [r * 0.0001 for r in total_recycled]
copper_tonnes = [r * 0.0009 for r in total_recycled]
iron_tonnes = [r * 0.999 for r in total_recycled]

# Q2: Waste sent to landfill in Q ending March 2020
# March 2020 = Q1 2020 = index 8 (0-based: 2018 has 4, 2019 has 4, then index 8)
q1_2020_idx = 8
landfill_q1_2020 = total_landfill[q1_2020_idx]
print(f'Landfill in Q ending Mar 2020: {landfill_q1_2020:,.0f} tonnes', file=sys.stderr)

print(f'\nWaste flow allocation complete.', file=sys.stderr)

In [ ]:
# ======================================================
# 5. INFLATION INDICES
# ======================================================
# Prices are given in 2017 prices.
# "Inflation should be applied such that a full year of inflation has been
# applied on or by 1 January of each subsequent year (i.e. for all inflation
# indices the value of the index on 1 January 2019 should reflect exactly
# one year of inflation)."
#
# This means:
#   Jan 1 2018: index = 1.0 (base, 2017 prices)
#   Jan 1 2019: index = (1+rate)^1  (one year of inflation)
#   Jan 1 2020: index = (1+rate)^2  (two years of inflation)
# So n = year - 2018 for annual stepping.

def annual_inflation_index(year, rate):
    """Annual stepping: same index for all quarters in a year.
    n = year - 2018. In 2018, prices are at base (2017) level."""
    n = year - 2018
    return (1 + rate) ** n

def quarterly_inflation_index(year, q, rate):
    """Quarterly stepping: inflation applied at start of each quarter.
    Jan 1 2018 (Q1 start) = 1.0 (base)
    Apr 1 2018 (Q2 start) = (1+rate)^(1/4)
    Jul 1 2018 (Q3 start) = (1+rate)^(2/4)
    Oct 1 2018 (Q4 start) = (1+rate)^(3/4)
    Jan 1 2019 (Q1 start) = (1+rate)^1 (exactly one year)
    """
    n = (year - 2018) + (q - 1) / 4
    return (1 + rate) ** n

# Build inflation indices for each quarter
idx_rev1 = []   # Rev 1: 2.5% annually
idx_rev2 = []   # Rev 2: 2% annually
idx_rev4 = []   # Rev 4 (Silver): 2.5% quarterly stepping
idx_rev5 = []   # Rev 5 (Copper): 2.5% quarterly stepping
idx_rev6 = []   # Rev 6 (Iron): 1.5% quarterly stepping
idx_cost1 = []  # Cost 1: 2% annually
idx_cost2 = []  # Cost 2: 2% annually
idx_cost3 = []  # Cost 3: 2% annually
idx_cost4 = []  # Cost 4: 5% annually

for i, qtr in enumerate(quarters):
    y = qtr['year']
    q = qtr['q']
    idx_rev1.append(annual_inflation_index(y, 0.025))
    idx_rev2.append(annual_inflation_index(y, 0.02))
    idx_rev4.append(quarterly_inflation_index(y, q, 0.025))
    idx_rev5.append(quarterly_inflation_index(y, q, 0.025))
    idx_rev6.append(quarterly_inflation_index(y, q, 0.015))
    idx_cost1.append(annual_inflation_index(y, 0.02))
    idx_cost2.append(annual_inflation_index(y, 0.02))
    idx_cost3.append(annual_inflation_index(y, 0.02))
    idx_cost4.append(annual_inflation_index(y, 0.05))

# Q6: Inflation index for Rev 4 (silver) in Q ending Sep 2026
# Sep 2026 = Q3 2026, index = (2026-2018)*4 + (3-1) = 34
q3_2026_idx = (2026 - 2018) * 4 + (3 - 1)  # = 34
rev4_idx_q3_2026 = idx_rev4[q3_2026_idx]
# (1.025)^8.5 = 1.23354 = 123.4% -> answer D
print(f'Rev 4 inflation index for Q3 2026: {rev4_idx_q3_2026 * 100:.1f}%', file=sys.stderr)

# Verify some indices
print(f'Rev 1 (annual 2.5%) Q1 2018: {idx_rev1[0]:.6f} (should be 1.0)', file=sys.stderr)
print(f'Rev 1 (annual 2.5%) Q1 2019: {idx_rev1[4]:.6f} (should be 1.025)', file=sys.stderr)
print(f'Rev 4 (quarterly 2.5%) Q1 2018: {idx_rev4[0]:.6f} (should be 1.0)', file=sys.stderr)
print(f'Rev 4 (quarterly 2.5%) Q2 2018: {idx_rev4[1]:.6f} (should be ~1.00621)', file=sys.stderr)
print(f'Rev 4 (quarterly 2.5%) Q1 2019: {idx_rev4[4]:.6f} (should be 1.025)', file=sys.stderr)

print('Inflation indices computed.', file=sys.stderr)

In [ ]:
# ======================================================
# 6. REVENUE CALCULATIONS
# ======================================================

# --- Rev 1: Third party waste gate fees ---
# $50/tonne in 2017 prices, 2.5% annual inflation
rev1 = [third_party[i] * 50 * idx_rev1[i] for i in range(n_quarters)]

# --- Rev 2: Guaranteed waste gate fees ---
# $45/tonne in 2017 prices, 2% annual inflation
# Minimum 75,000 tonnes per quarter (pay for 75,000 even if less received)
rev2_tonnage = [max(g, 75000) for g in guaranteed]
rev2 = [rev2_tonnage[i] * 45 * idx_rev2[i] for i in range(n_quarters)]

# Overcharged tonnage (difference between billed and actual when actual < 75,000)
overcharged = [max(75000 - g, 0) for g in guaranteed]

print(f'Total Rev 2 (guaranteed gate fees): ${sum(rev2):,.0f}', file=sys.stderr)
print(f'Total overcharged tonnage: {sum(overcharged):,.0f}', file=sys.stderr)

# --- Rev 3: Additional waste gate fees ---
# $70/tonne, NOT inflated
# The tonnage for Rev 3 is reduced by overcharged tonnage from Rev 2
# Overcharged tonnage from period i is deducted in NEXT periods (i+1 onwards)
# Overcharged tonnage carried forward until fully written off
# Rev 3 has a payment delay of one quarter

carry_forward = 0.0  # accumulated overcharged tonnage
rev3_effective_tonnage = []
for i in range(n_quarters):
    # First, deduct any accumulated carry-forward from additional waste
    consumed = min(carry_forward, additional_actual[i])
    effective = additional_actual[i] - consumed
    carry_forward -= consumed
    rev3_effective_tonnage.append(effective)
    # Then add any new overcharge from this quarter (applies from next period)
    carry_forward += overcharged[i]

total_rev3_tonnage = sum(rev3_effective_tonnage)
print(f'Total Rev 3 effective tonnage: {total_rev3_tonnage:,.2f}', file=sys.stderr)
print(f'Remaining carry-forward: {carry_forward:,.2f}', file=sys.stderr)

# Print quarters where overcharge or carry-forward reduction happens
for i in range(n_quarters):
    if overcharged[i] > 0 or (additional_actual[i] > 0 and rev3_effective_tonnage[i] < additional_actual[i]):
        print(f'  {quarters[i]["label"]}: guaranteed={guaranteed[i]:,.0f}, additional={additional_actual[i]:,.0f}, '
              f'overcharged={overcharged[i]:,.0f}, rev3_eff={rev3_effective_tonnage[i]:,.0f}', file=sys.stderr)

# Rev 3 revenue (before payment delay) = effective_tonnage * $70
rev3_before_delay = [t * 70 for t in rev3_effective_tonnage]

# Payment delay of one quarter: Rev 3 for quarter i is received in quarter i+1
# No revenue for waste received prior to the start of the modelled timeline
rev3 = [0.0] * n_quarters
for i in range(1, n_quarters):
    rev3[i] = rev3_before_delay[i - 1]

# --- Rev 4: Silver recycling ---
# $18/oz in 2017 prices, 2.5% quarterly inflation
# 35,274 oz per tonne
OZ_PER_TONNE = 35274
rev4 = [silver_tonnes[i] * OZ_PER_TONNE * 18 * idx_rev4[i] for i in range(n_quarters)]

# --- Rev 5: Copper recycling ---
# $3/oz in 2017 prices, 2.5% quarterly inflation
rev5 = [copper_tonnes[i] * OZ_PER_TONNE * 3 * idx_rev5[i] for i in range(n_quarters)]

# --- Rev 6: Iron recycling ---
# $150/tonne in 2017 prices, 1.5% quarterly inflation
rev6 = [iron_tonnes[i] * 150 * idx_rev6[i] for i in range(n_quarters)]

print(f'\nRev 5 (copper) in Q4 2026: ${rev5[(2026-2018)*4+3]:,.2f}', file=sys.stderr)
print(f'Total Rev 4 (silver): ${sum(rev4):,.0f}', file=sys.stderr)
print(f'Total Rev 5 (copper): ${sum(rev5):,.0f}', file=sys.stderr)
print(f'Total Rev 6 (iron): ${sum(rev6):,.0f}', file=sys.stderr)

print('\nRevenue calculations complete.', file=sys.stderr)

In [ ]:
# ======================================================
# 7. COST CALCULATIONS
# ======================================================

# --- Cost 1: Incinerator running costs ---
# $70,000/year in 2017 prices, 2% annually, paid in May
# May falls in Q2 (Apr-Jun) of each year
cost1 = [0.0] * n_quarters
for i, qtr in enumerate(quarters):
    if qtr['q'] == 2:  # Q2 contains May
        cost1[i] = 70000 * idx_cost1[i]

# --- Cost 2: Incinerator processing costs ---
# $20/tonne incinerated in 2017 prices, 2% annually
cost2 = [total_incinerated[i] * 20 * idx_cost2[i] for i in range(n_quarters)]

# Q8: Total Cost 2 from Jan 2018 to Dec 2021
# Q1 2018 through Q4 2021 = indices 0 through 15
cost2_2018_2021 = sum(cost2[:16])
print(f'Total Cost 2 (2018-2021): ${cost2_2018_2021:,.0f}', file=sys.stderr)

# --- Cost 3: Landfill costs ---
# $150/tonne landfilled in 2017 prices, 2% annually
cost3 = [total_landfill[i] * 150 * idx_cost3[i] for i in range(n_quarters)]

# --- Cost 4: Landfill penalty ---
# $500,000/year in 2017 prices, 5% annually
# Paid in December if total waste landfilled in current year > 65,500 tonnes

# Calculate annual landfill totals
cost4 = [0.0] * n_quarters
penalty_count = 0

for year in range(2018, 2031):
    # Sum landfill for all 4 quarters of this year
    year_start_idx = (year - 2018) * 4
    annual_landfill = sum(total_landfill[year_start_idx:year_start_idx + 4])
    
    if annual_landfill > 65500:
        # Penalty paid in December = Q4 of this year
        q4_idx = year_start_idx + 3
        cost4[q4_idx] = 500000 * idx_cost4[q4_idx]
        penalty_count += 1
        print(f'  Penalty in {year}: annual landfill = {annual_landfill:,.0f} tonnes, penalty = ${cost4[q4_idx]:,.0f}', file=sys.stderr)
    else:
        print(f'  No penalty in {year}: annual landfill = {annual_landfill:,.0f} tonnes', file=sys.stderr)

print(f'\nLandfill penalty count: {penalty_count}', file=sys.stderr)

print('Cost calculations complete.', file=sys.stderr)

In [ ]:
# ======================================================
# 8. ANSWER ALL QUESTIONS
# ======================================================

def match_mc(question_file, value):
    """Parse question file and match computed value to closest MC option."""
    with open(question_file) as f:
        text = f.read()
    options = {}
    # Match patterns like "A. 82,948" or "A. $198,184,476" or "A. 123.1%"
    for m in re.finditer(r'([A-I])\.\s+([\$\d,\.\-\%]+)', text):
        letter = m.group(1)
        cleaned = m.group(2).strip().replace('$', '').replace(',', '').replace('%', '')
        try:
            options[letter] = float(cleaned)
        except ValueError:
            pass
    if not options:
        print(f'  WARNING: No MC options found in {question_file}', file=sys.stderr)
        return None
    best = min(options, key=lambda k: abs(options[k] - value))
    diff = abs(options[best] - value)
    print(f'  MC match: value={value:,.4f}, best={best} ({options[best]:,.1f}), diff={diff:,.4f}', file=sys.stderr)
    if diff > 10:
        print(f'  WARNING: Large diff! All options: {options}', file=sys.stderr)
    return best

# Question 1: Total additional waste received (tonnes)
q1_val = total_additional
q1 = match_mc(f'{DATA_DIR}/question1.txt', q1_val)
print(f'Q1: Total additional waste = {q1_val:,.2f} -> {q1}', file=sys.stderr)

# Question 2: Landfill in Q ending March 2020 (Q1 2020, index 8)
q2_val = total_landfill[q1_2020_idx]
q2 = match_mc(f'{DATA_DIR}/question2.txt', q2_val)
print(f'Q2: Landfill Q1 2020 = {q2_val:,.2f} -> {q2}', file=sys.stderr)

# Question 3: Total Rev 2 (guaranteed waste gate fees)
q3_val = sum(rev2)
q3 = match_mc(f'{DATA_DIR}/question3.txt', q3_val)
print(f'Q3: Total Rev 2 = ${q3_val:,.2f} -> {q3}', file=sys.stderr)

# Question 4: Total tonnage for Rev 3 (after adjustment, before payment timing)
q4_val = total_rev3_tonnage
q4 = match_mc(f'{DATA_DIR}/question4.txt', q4_val)
print(f'Q4: Total Rev 3 tonnage = {q4_val:,.2f} -> {q4}', file=sys.stderr)

# Question 5: Cash from gate fees (Rev 1 + Rev 2 + Rev 3) in Q ending Sep 2024
# Sep 2024 = Q3 2024 = index (2024-2018)*4 + 2 = 26
q3_2024_idx = (2024 - 2018) * 4 + 2  # = 26
print(f'  Q5 quarter: {quarters[q3_2024_idx]["label"]}', file=sys.stderr)
q5_rev1 = rev1[q3_2024_idx]
q5_rev2 = rev2[q3_2024_idx]
q5_rev3 = rev3[q3_2024_idx]  # This is cash received (includes delay)
q5_val = q5_rev1 + q5_rev2 + q5_rev3
print(f'  Rev 1: ${q5_rev1:,.2f}, Rev 2: ${q5_rev2:,.2f}, Rev 3: ${q5_rev3:,.2f}', file=sys.stderr)
q5 = match_mc(f'{DATA_DIR}/question5.txt', q5_val)
print(f'Q5: Gate fees Q3 2024 = ${q5_val:,.2f} -> {q5}', file=sys.stderr)

# Question 6: Inflation index for Rev 4 in Q ending Sep 2026
q6_val_pct = rev4_idx_q3_2026 * 100
q6 = match_mc(f'{DATA_DIR}/question6.txt', q6_val_pct)
print(f'Q6: Rev 4 inflation Q3 2026 = {q6_val_pct:.2f}% -> {q6}', file=sys.stderr)

# Question 7: Rev 5 (copper recycling) in Q ending Dec 2026
q4_2026_idx = (2026 - 2018) * 4 + 3  # = 35
print(f'  Q7 quarter: {quarters[q4_2026_idx]["label"]}', file=sys.stderr)
q7_val = rev5[q4_2026_idx]
q7 = match_mc(f'{DATA_DIR}/question7.txt', q7_val)
print(f'Q7: Rev 5 Q4 2026 = ${q7_val:,.2f} -> {q7}', file=sys.stderr)

# Question 8: Total Cost 2 from Jan 2018 to Dec 2021
q8_val = cost2_2018_2021
q8 = match_mc(f'{DATA_DIR}/question8.txt', q8_val)
print(f'Q8: Total Cost 2 (2018-2021) = ${q8_val:,.2f} -> {q8}', file=sys.stderr)

# Question 9: How many times is landfill penalty paid? (free-form answer)
q9_val = str(penalty_count)
print(f'Q9: Landfill penalty count = {q9_val}', file=sys.stderr)

# Question 10: Total cash revenues less total costs in year 2027 (free-form)
year_2027_start = (2027 - 2018) * 4  # 36
year_2027_end = year_2027_start + 4    # 40 (exclusive)

total_rev_2027 = 0
total_cost_2027 = 0
print(f'  Year 2027 quarters: {quarters[year_2027_start]["label"]} to {quarters[year_2027_end-1]["label"]}', file=sys.stderr)
for i in range(year_2027_start, year_2027_end):
    qrev = rev1[i] + rev2[i] + rev3[i] + rev4[i] + rev5[i] + rev6[i]
    qcost = cost1[i] + cost2[i] + cost3[i] + cost4[i]
    total_rev_2027 += qrev
    total_cost_2027 += qcost
    print(f'    {quarters[i]["label"]}: rev=${qrev:,.0f}, cost=${qcost:,.0f}', file=sys.stderr)

q10_val = total_rev_2027 - total_cost_2027
q10_str = f'${round(q10_val)}'
print(f'Q10: Rev 2027: ${total_rev_2027:,.0f}, Cost 2027: ${total_cost_2027:,.0f}', file=sys.stderr)
print(f'Q10: Net cash 2027 = {q10_str}', file=sys.stderr)

print('\nAll questions answered.', file=sys.stderr)

In [ ]:
# ======================================================
# 9. SET ANSWERS
# ======================================================

answers = {
    'question1': q1,
    'question2': q2,
    'question3': q3,
    'question4': q4,
    'question5': q5,
    'question6': q6,
    'question7': q7,
    'question8': q8,
    'question9': q9_val,
    'question10': q10_str,
}

print('\n' + '=' * 60, file=sys.stderr)
print('Final answers:', file=sys.stderr)
for k, v in answers.items():
    print(f'  {k}: {v}', file=sys.stderr)
print('=' * 60, file=sys.stderr)